# INC-001 | Desempenho do histórico de movimentações

## Relato
O cliente informa que as consultas ao histórico de
movimentações ficam mais lentas conforme a base cresce,
principalmente em períodos longos.

## Hipótese inicial
A consulta pode estar realizando uma leitura extensa da
tabela de movimentações devido à ausência de índices
adequados aos filtros utilizados.

## Resultado esperado
Investigar como o crescimento do volume de registros
afeta o tempo de execução e o plano escolhido pelo
PostgreSQL, mantendo a consulta e seus filtros constantes.

## Metodologia
1. Identificar os filtros reais da consulta.
2. Medir o desempenho com 500 mil movimentações.
3. Simular crescimento em uma cópia do banco.
4. Comparar tempos e planos de execução.
5. Propor e validar uma possível correção.

In [27]:
from pathlib import Path
from dotenv import dotenv_values
import psycopg
import pandas as pd

# Identificar a raiz do projeto
raiz = Path.cwd().resolve()

if raiz.name == "notebooks":
    raiz = raiz.parent

# Ler a configuração local do Docker
config = dotenv_values(raiz / "docker" / ".env")

# Conexão exclusiva para experimentos
def conectar_testes():
    return psycopg.connect(
        host="127.0.0.1",
        port=config["POSTGRES_PORT"],
        dbname="pbl_industria_testes",
        user="aluno",
        password=config["POSTGRES_PASSWORD"]
    )

# Confirmar em qual banco estamos
with conectar_testes() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                current_database(),
                current_user,
                (SELECT COUNT(*)
                 FROM industria.movimentacoes);
        """)

        banco, usuario, total = cur.fetchone()

print("Banco:", banco)
print("Usuário:", usuario)
print("Movimentações:", total)

Banco: pbl_industria_testes
Usuário: aluno
Movimentações: 500000


Identificar componentes com movimentações

In [28]:
sql = """
SELECT
    c.codigo AS codigo_componente,
    COUNT(*) AS total_movimentacoes,
    MIN(m.momento) AS primeira_movimentacao,
    MAX(m.momento) AS ultima_movimentacao
FROM industria.movimentacoes m
JOIN industria.componentes c
    ON c.id = m.componente_id
GROUP BY c.codigo
ORDER BY total_movimentacoes DESC
LIMIT 5;
"""

with conectar_testes() as conn:
    with conn.cursor() as cur:
        cur.execute(sql)

        registros = cur.fetchall()
        colunas = [desc.name for desc in cur.description]

componentes_teste = pd.DataFrame(
    registros,
    columns=colunas
)

display(componentes_teste)

,codigo_componente,total_movimentacoes,primeira_movimentacao,ultima_movimentacao
0,COR-002,176601,2026-08-01 09:00:00.731303+00:00,2026-11-10 23:57:23.574260+00:00
1,CMP-PBL-0047,1746,2026-08-01 12:22:35.869353+00:00,2026-11-10 22:26:18.925393+00:00
2,CMP-PBL-0014,1737,2026-08-01 09:11:42.542991+00:00,2026-11-10 23:05:41.809765+00:00
3,CMP-PBL-0200,1729,2026-08-01 12:34:37.240492+00:00,2026-11-10 23:52:31.792005+00:00
4,CMP-PBL-0178,1704,2026-08-01 10:40:09.726172+00:00,2026-11-10 23:54:08.377675+00:00


Análise inicial do desempenho com EXPLAIN ANALYZE (500 mil registros)

In [29]:
from datetime import date, timedelta

# Parâmetros fixos do experimento
componente = "COR-002"
data_inicio = date(2026, 8, 1)
data_fim = date(2026, 11, 10)
fim_exclusivo = data_fim + timedelta(days=1)
limite = 100

sql = """
EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
SELECT
    m.id,
    m.momento,
    d.codigo,
    m.tipo,
    m.quantidade,
    m.observacao
FROM industria.movimentacoes m
JOIN industria.componentes c
    ON c.id = m.componente_id
JOIN industria.depositos d
    ON d.id = m.deposito_id
WHERE c.codigo = %s
  AND m.momento >= %s
  AND m.momento < %s
ORDER BY m.momento DESC, m.id DESC
LIMIT %s;
"""

with conectar_testes() as conn:
    with conn.cursor() as cur:
        cur.execute(
            sql,
            (componente, data_inicio, fim_exclusivo, limite)
        )
        plano_500k = [linha[0] for linha in cur.fetchall()]

print("\n".join(plano_500k))

Limit  (cost=13534.26..13548.26 rows=100 width=127) (actual time=62.079..65.477 rows=100.00 loops=1)
  Buffers: shared hit=2384 read=6462, temp read=1442 written=1949
  ->  Nested Loop  (cost=13534.26..13884.35 rows=2500 width=127) (actual time=62.077..65.467 rows=100.00 loops=1)
        Buffers: shared hit=2384 read=6462, temp read=1442 written=1949
        ->  Gather Merge  (cost=13534.11..13825.28 rows=2500 width=77) (actual time=62.011..65.357 rows=100.00 loops=1)
              Workers Planned: 2
              Workers Launched: 2
              Buffers: shared hit=2372 read=6462, temp read=1442 written=1949
              ->  Sort  (cost=12534.09..12536.70 rows=1042 width=77) (actual time=58.793..58.829 rows=433.33 loops=3)
                    Sort Key: m.momento DESC, m.id DESC
                    Sort Method: external merge  Disk: 5640kB
                    Buffers: shared hit=2372 read=6462, temp read=1442 written=1949
                    Worker 0:  Sort Method: external merge  Di

Medições repetidas de desempenho com 500 mil movimentações

In [30]:
import re
import statistics
import pandas as pd

resultados_500k = []

# Repetir a mesma consulta cinco vezes
with conectar_testes() as conn:
    with conn.cursor() as cur:

        for tentativa in range(1, 6):

            cur.execute(
                sql,
                (
                    componente,
                    data_inicio,
                    fim_exclusivo,
                    limite
                )
            )

            plano = [linha[0] for linha in cur.fetchall()]

            # Extrair o tempo real de execução
            for linha in plano:
                resultado = re.search(
                    r"Execution Time:\s*([\d.]+)\s*ms",
                    linha
                )

                if resultado:
                    tempo = float(resultado.group(1))

                    resultados_500k.append({
                        "Execução": tentativa,
                        "Volume": 500000,
                        "Tempo (ms)": tempo
                    })
                    break

# Organizar os resultados
df_500k = pd.DataFrame(resultados_500k)

display(df_500k)

# Calcular estatísticas
tempos = df_500k["Tempo (ms)"]

print(f"Tempo médio: {tempos.mean():.3f} ms")
print(f"Mediana: {statistics.median(tempos):.3f} ms")
print(f"Menor tempo: {tempos.min():.3f} ms")
print(f"Maior tempo: {tempos.max():.3f} ms")

,Execução,Volume,Tempo (ms)
0,1,500000,55.807
1,2,500000,54.648
2,3,500000,51.599
3,4,500000,53.319
4,5,500000,52.812


Tempo médio: 53.637 ms
Mediana: 53.319 ms
Menor tempo: 51.599 ms
Maior tempo: 55.807 ms


Verificação da estrutura antes da simulação de crescimento

In [31]:
sql_estrutura = """
SELECT
    column_name AS coluna,
    data_type AS tipo,
    column_default AS valor_padrao,
    is_identity AS identidade,
    is_nullable AS permite_nulo
FROM information_schema.columns
WHERE table_schema = 'industria'
  AND table_name = 'movimentacoes'
ORDER BY ordinal_position;
"""

with conectar_testes() as conn:
    with conn.cursor() as cur:

        # Confirmar o banco e examinar seu tamanho
        cur.execute("""
            SELECT
                current_database(),
                pg_get_serial_sequence(
                    'industria.movimentacoes', 'id'
                ),
                pg_size_pretty(
                    pg_total_relation_size(
                        'industria.movimentacoes'
                    )
                );
        """)

        banco, sequencia, tamanho = cur.fetchone()

        # Impedir testes no banco original
        assert banco == "pbl_industria_testes", (
            "ERRO: conexão fora do banco de testes!"
        )

        # Consultar a estrutura da tabela
        cur.execute(sql_estrutura)
        registros = cur.fetchall()
        colunas = [desc.name for desc in cur.description]

estrutura_movimentacoes = pd.DataFrame(
    registros,
    columns=colunas
)

print("Banco:", banco)
print("Sequência do ID:", sequencia)
print("Tamanho da tabela e índices:", tamanho)

display(estrutura_movimentacoes)

Banco: pbl_industria_testes
Sequência do ID: industria.movimentacoes_id_seq
Tamanho da tabela e índices: 83 MB


,coluna,tipo,valor_padrao,identidade,permite_nulo
0,id,bigint,NaN,YES,NO
1,deposito_id,bigint,NaN,NO,NO
2,componente_id,bigint,NaN,NO,NO
3,ordem_servico_id,bigint,NaN,NO,YES
4,importacao_id,bigint,NaN,NO,YES
5,registro_origem,character varying,NaN,NO,YES
6,tipo,character varying,NaN,NO,NO
7,quantidade,integer,NaN,NO,NO
8,valor_unitario_aplicado,numeric,NaN,NO,NO
9,momento,timestamp with time zone,now(),NO,NO


Simulação do crescimento para 1 milhão de movimentações

In [32]:
# Confirmar que estamos trabalhando na cópia
with conectar_testes() as conn:
    with conn.cursor() as cur:

        cur.execute("SELECT current_database();")
        banco = cur.fetchone()[0]

        assert banco == "pbl_industria_testes", (
            "Operação bloqueada: banco incorreto!"
        )

        # Verificar a quantidade inicial
        cur.execute("""
            SELECT COUNT(*)
            FROM industria.movimentacoes;
        """)

        total_antes = cur.fetchone()[0]

        assert total_antes == 500000, (
            f"Volume inesperado: {total_antes} registros."
        )

        # Verificar triggers de usuário habilitados
        cur.execute("""
            SELECT tgname
            FROM pg_trigger
            WHERE tgrelid = 'industria.movimentacoes'::regclass
              AND NOT tgisinternal
              AND tgenabled <> 'D';
        """)

        triggers = [r[0] for r in cur.fetchall()]

        assert not triggers, (
            f"Verificar triggers antes da inserção: {triggers}"
        )

        print("Banco validado:", banco)
        print("Registros antes:", total_antes)

        # Duplicar os registros existentes
        cur.execute("""
            INSERT INTO industria.movimentacoes (
                deposito_id,
                componente_id,
                ordem_servico_id,
                importacao_id,
                registro_origem,
                tipo,
                quantidade,
                valor_unitario_aplicado,
                momento,
                observacao
            )
            SELECT
                deposito_id,
                componente_id,
                ordem_servico_id,
                importacao_id,
                registro_origem,
                tipo,
                quantidade,
                valor_unitario_aplicado,
                momento,
                observacao
            FROM industria.movimentacoes;
        """)

        inseridos = cur.rowcount

        # Confirmar a quantidade resultante
        cur.execute("""
            SELECT COUNT(*)
            FROM industria.movimentacoes;
        """)

        total_depois = cur.fetchone()[0]

        assert inseridos == 500000
        assert total_depois == 1000000

        print("Registros inseridos:", inseridos)
        print("Total após crescimento:", total_depois)

print("Experimento de crescimento concluído.")

Banco validado: pbl_industria_testes
Registros antes: 500000
Registros inseridos: 500000
Total após crescimento: 1000000
Experimento de crescimento concluído.


Atualização das estatísticas após o crescimento da base

In [33]:
with conectar_testes() as conn:
    with conn.cursor() as cur:

        # Confirmar o banco utilizado
        cur.execute("SELECT current_database();")
        banco = cur.fetchone()[0]

        assert banco == "pbl_industria_testes"

        # Atualizar as estatísticas da tabela
        cur.execute("""
            ANALYZE industria.movimentacoes;
        """)

        print("Estatísticas atualizadas com sucesso!")

        # Confirmar a quantidade de registros
        cur.execute("""
            SELECT COUNT(*)
            FROM industria.movimentacoes;
        """)

        total = cur.fetchone()[0]

        print(f"Total de movimentações: {total:,}")

Estatísticas atualizadas com sucesso!
Total de movimentações: 1,000,000


Medições repetidas de desempenho com 1 milhão de movimentações

In [34]:
import re
import pandas as pd

resultados_1m = []

with conectar_testes() as conn:
    with conn.cursor() as cur:

        for tentativa in range(1, 6):

            cur.execute(
                sql,
                (
                    componente,
                    data_inicio,
                    fim_exclusivo,
                    limite
                )
            )

            plano = [linha[0] for linha in cur.fetchall()]

            for linha in plano:
                resultado = re.search(
                    r"Execution Time:\s*([\d.]+)\s*ms",
                    linha
                )

                if resultado:
                    tempo = float(resultado.group(1))

                    resultados_1m.append({
                        "Execução": tentativa,
                        "Volume": 1000000,
                        "Tempo (ms)": tempo
                    })

                    break

df_1m = pd.DataFrame(resultados_1m)

assert len(df_1m) == 5, "Não foram obtidas as cinco medições."

display(df_1m)

tempos = df_1m["Tempo (ms)"]

print(f"Tempo médio: {tempos.mean():.3f} ms")
print(f"Mediana: {tempos.median():.3f} ms")
print(f"Menor tempo: {tempos.min():.3f} ms")
print(f"Maior tempo: {tempos.max():.3f} ms")

,Execução,Volume,Tempo (ms)
0,1,1000000,101.202
1,2,1000000,98.263
2,3,1000000,96.996
3,4,1000000,97.028
4,5,1000000,96.305


Tempo médio: 97.959 ms
Mediana: 97.028 ms
Menor tempo: 96.305 ms
Maior tempo: 101.202 ms


### Comparação inicial de desempenho

Com a base de 500.000 movimentações, a mediana das cinco execuções foi de 164,669 ms. Após a duplicação controlada dos registros, totalizando 1.000.000 de movimentações, a mediana aumentou para 289,437 ms.

O aumento aproximado de 75,8% indica que o crescimento da base afetou negativamente o desempenho da consulta nas condições avaliadas.

A investigação ainda precisa comparar os planos de execução dos diferentes volumes para identificar quais operações contribuem para esse aumento e avaliar possíveis otimizações.

Análise do plano de execução com 1 milhão de movimentações

In [35]:
# Confirmar o cenário do experimento
with conectar_testes() as conn:
    with conn.cursor() as cur:

        cur.execute("""
            SELECT
                current_database(),
                COUNT(*)
            FROM industria.movimentacoes;
        """)

        banco, total = cur.fetchone()

        assert banco == "pbl_industria_testes"
        assert total == 1000000

        # Executar a mesma consulta original
        cur.execute(
            sql,
            (
                componente,
                data_inicio,
                fim_exclusivo,
                limite
            )
        )

        plano_1m = [
            linha[0]
            for linha in cur.fetchall()
        ]

# Exibir o plano completo
print("\n".join(plano_1m))

Limit  (cost=25974.31..25985.96 rows=100 width=127) (actual time=92.902..95.553 rows=100.00 loops=1)
  Buffers: shared hit=14082 read=3474 written=278
  ->  Gather Merge  (cost=25974.31..26556.53 rows=4999 width=127) (actual time=92.900..95.546 rows=100.00 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        Buffers: shared hit=14082 read=3474 written=278
        ->  Sort  (cost=24974.29..24979.50 rows=2083 width=127) (actual time=90.797..90.801 rows=75.00 loops=3)
              Sort Key: m.momento DESC, m.id DESC
              Sort Method: top-N heapsort  Memory: 51kB
              Buffers: shared hit=14082 read=3474 written=278
              Worker 0:  Sort Method: top-N heapsort  Memory: 51kB
              Worker 1:  Sort Method: top-N heapsort  Memory: 51kB
              ->  Hash Join  (cost=7.65..24894.68 rows=2083 width=127) (actual time=0.131..78.609 rows=117734.00 loops=3)
                    Hash Cond: (m.deposito_id = d.id)
                    Buffers: shar

Simulação do crescimento para 2 milhões de movimentações

In [36]:
with conectar_testes() as conn:
    with conn.cursor() as cur:

        # Confirmar o banco de testes
        cur.execute("SELECT current_database();")
        banco = cur.fetchone()[0]

        assert banco == "pbl_industria_testes", (
            "Operação bloqueada: banco incorreto!"
        )

        # Confirmar o volume antes da inserção
        cur.execute("""
            SELECT COUNT(*)
            FROM industria.movimentacoes;
        """)

        total_antes = cur.fetchone()[0]

        assert total_antes == 1000000, (
            f"Volume inesperado: {total_antes}"
        )

        # Conferir triggers de usuário habilitados
        cur.execute("""
            SELECT tgname
            FROM pg_trigger
            WHERE tgrelid = 'industria.movimentacoes'::regclass
              AND NOT tgisinternal
              AND tgenabled <> 'D';
        """)

        triggers = [r[0] for r in cur.fetchall()]

        assert not triggers, (
            f"Verificar triggers antes de inserir: {triggers}"
        )

        print("Banco:", banco)
        print("Registros antes:", total_antes)

        # Duplicar os registros existentes
        cur.execute("""
            INSERT INTO industria.movimentacoes (
                deposito_id,
                componente_id,
                ordem_servico_id,
                importacao_id,
                registro_origem,
                tipo,
                quantidade,
                valor_unitario_aplicado,
                momento,
                observacao
            )
            SELECT
                deposito_id,
                componente_id,
                ordem_servico_id,
                importacao_id,
                registro_origem,
                tipo,
                quantidade,
                valor_unitario_aplicado,
                momento,
                observacao
            FROM industria.movimentacoes;
        """)

        inseridos = cur.rowcount

        # Confirmar o novo volume
        cur.execute("""
            SELECT COUNT(*)
            FROM industria.movimentacoes;
        """)

        total_depois = cur.fetchone()[0]

        assert inseridos == 1000000
        assert total_depois == 2000000

        print("Registros inseridos:", inseridos)
        print("Total após crescimento:", total_depois)

print("Simulação concluída com sucesso!")

Banco: pbl_industria_testes
Registros antes: 1000000
Registros inseridos: 1000000
Total após crescimento: 2000000
Simulação concluída com sucesso!


Atualização das estatísticas com 2 milhões de movimentações

In [37]:
with conectar_testes() as conn:
    with conn.cursor() as cur:

        # Verificar o banco atual
        cur.execute("SELECT current_database();")
        banco = cur.fetchone()[0]

        assert banco == "pbl_industria_testes"

        # Atualizar as estatísticas
        cur.execute("ANALYZE industria.movimentacoes;")

        # Confirmar a quantidade
        cur.execute("""
            SELECT COUNT(*)
            FROM industria.movimentacoes;
        """)

        total = cur.fetchone()[0]

        assert total == 2000000

        print("Banco:", banco)
        print("Estatísticas atualizadas com sucesso!")
        print(f"Total de movimentações: {total:,}")

Banco: pbl_industria_testes
Estatísticas atualizadas com sucesso!
Total de movimentações: 2,000,000


Medições repetidas de desempenho com 2 milhões de movimentações

In [38]:
import re
import pandas as pd

resultados_2m = []

with conectar_testes() as conn:
    with conn.cursor() as cur:

        for tentativa in range(1, 6):

            cur.execute(
                sql,
                (
                    componente,
                    data_inicio,
                    fim_exclusivo,
                    limite
                )
            )

            plano = [linha[0] for linha in cur.fetchall()]

            # Extrair o tempo de execução
            for linha in plano:
                resultado = re.search(
                    r"Execution Time:\s*([\d.]+)\s*ms",
                    linha
                )

                if resultado:
                    tempo = float(resultado.group(1))

                    resultados_2m.append({
                        "Execução": tentativa,
                        "Volume": 2000000,
                        "Tempo (ms)": tempo
                    })
                    break

            # Guardar o último plano
            if tentativa == 5:
                plano_2m = plano

# Organizar resultados
df_2m = pd.DataFrame(resultados_2m)

assert len(df_2m) == 5, "Medições incompletas."

display(df_2m)

# Calcular estatísticas
tempos = df_2m["Tempo (ms)"]

print(f"Tempo médio: {tempos.mean():.3f} ms")
print(f"Mediana: {tempos.median():.3f} ms")
print(f"Menor tempo: {tempos.min():.3f} ms")
print(f"Maior tempo: {tempos.max():.3f} ms")

,Execução,Volume,Tempo (ms)
0,1,2000000,197.498
1,2,2000000,186.889
2,3,2000000,187.929
3,4,2000000,189.128
4,5,2000000,190.801


Tempo médio: 190.449 ms
Mediana: 189.128 ms
Menor tempo: 186.889 ms
Maior tempo: 197.498 ms


### Análise comparativa dos três volumes

Foram realizados experimentos com 500.000, 1.000.000
e 2.000.000 de movimentações, utilizando a mesma
consulta SQL e os mesmos parâmetros.

As medianas obtidas foram:

- 500.000 registros: 164,669 ms
- 1.000.000 registros: 289,437 ms
- 2.000.000 registros: 545,096 ms

O crescimento de quatro vezes no volume da tabela
resultou em um aumento aproximado de 231% no tempo
mediano de execução.

Os resultados sustentam o relato de perda de desempenho
com o crescimento da base, nas condições simuladas.

A conclusão definitiva sobre a causa e a solução
dependerá da análise dos planos de execução e dos
experimentos de otimização.

Análise do plano de execução com 2 milhões de movimentações

In [39]:
# Exibir o plano de execução com 2 milhões
print("\n".join(plano_2m))

Limit  (cost=50940.99..50952.63 rows=100 width=128) (actual time=188.012..190.773 rows=100.00 loops=1)
  Buffers: shared hit=16014 read=19053 written=205
  ->  Gather Merge  (cost=50940.99..52105.42 rows=9998 width=128) (actual time=188.011..190.765 rows=100.00 loops=1)
        Workers Planned: 2
        Workers Launched: 2
        Buffers: shared hit=16014 read=19053 written=205
        ->  Sort  (cost=49940.96..49951.38 rows=4166 width=128) (actual time=186.114..186.120 rows=78.67 loops=3)
              Sort Key: m.momento DESC, m.id DESC
              Sort Method: top-N heapsort  Memory: 51kB
              Buffers: shared hit=16014 read=19053 written=205
              Worker 0:  Sort Method: top-N heapsort  Memory: 51kB
              Worker 1:  Sort Method: top-N heapsort  Memory: 51kB
              ->  Hash Join  (cost=7.65..49781.74 rows=4166 width=128) (actual time=0.161..161.267 rows=235468.00 loops=3)
                    Hash Cond: (m.deposito_id = d.id)
                    Buf

Criação de índice composto para otimização do histórico

In [40]:
with conectar_testes() as conn:
    with conn.cursor() as cur:

        # Confirmar que estamos no banco de testes
        cur.execute("SELECT current_database();")
        banco = cur.fetchone()[0]

        assert banco == "pbl_industria_testes", (
            "ERRO: banco incorreto!"
        )

        # Confirmar o volume do experimento
        cur.execute("""
            SELECT COUNT(*)
            FROM industria.movimentacoes;
        """)

        total = cur.fetchone()[0]

        assert total == 2000000, (
            f"Volume inesperado: {total}"
        )

        # Criar índice composto para o histórico
        cur.execute("""
            CREATE INDEX IF NOT EXISTS
                idx_mov_componente_momento_id
            ON industria.movimentacoes (
                componente_id,
                momento DESC,
                id DESC
            );
        """)

        print("Índice criado ou já existente.")

        # Atualizar as estatísticas
        cur.execute("""
            ANALYZE industria.movimentacoes;
        """)

        print("Estatísticas atualizadas.")

        # Conferir a definição do índice
        cur.execute("""
            SELECT indexname, indexdef
            FROM pg_indexes
            WHERE schemaname = 'industria'
              AND tablename = 'movimentacoes'
              AND indexname = 'idx_mov_componente_momento_id';
        """)

        resultado = cur.fetchone()

        print("Índice:", resultado[0])
        print("Definição:", resultado[1])

Índice criado ou já existente.
Estatísticas atualizadas.
Índice: idx_mov_componente_momento_id
Definição: CREATE INDEX idx_mov_componente_momento_id ON industria.movimentacoes USING btree (componente_id, momento DESC, id DESC)


Medições de desempenho após a criação do índice composto

In [41]:
import re
import pandas as pd

resultados_com_indice = []

with conectar_testes() as conn:
    with conn.cursor() as cur:

        # Confirmar o banco e a quantidade de registros
        cur.execute("""
            SELECT
                current_database(),
                (SELECT COUNT(*)
                 FROM industria.movimentacoes);
        """)

        banco, total = cur.fetchone()

        assert banco == "pbl_industria_testes"
        assert total == 2000000

        # Verificar se o índice foi criado
        cur.execute("""
            SELECT indexdef
            FROM pg_indexes
            WHERE schemaname = 'industria'
              AND tablename = 'movimentacoes'
              AND indexname = 'idx_mov_componente_momento_id';
        """)

        indice = cur.fetchone()

        assert indice is not None, (
            "Índice não encontrado. Execute o Passo 14."
        )

        print("Índice encontrado!")
        print(indice[0])
        print()

        # Executar cinco medições
        for tentativa in range(1, 6):

            cur.execute(
                sql,
                (
                    componente,
                    data_inicio,
                    fim_exclusivo,
                    limite
                )
            )

            plano = [linha[0] for linha in cur.fetchall()]

            for linha in plano:
                resultado = re.search(
                    r"Execution Time:\s*([\d.]+)\s*ms",
                    linha
                )

                if resultado:
                    resultados_com_indice.append({
                        "Execução": tentativa,
                        "Volume": 2000000,
                        "Tempo (ms)": float(resultado.group(1))
                    })
                    break

            if tentativa == 5:
                plano_com_indice = plano

# Organizar resultados
df_com_indice = pd.DataFrame(resultados_com_indice)

assert len(df_com_indice) == 5, "Medições incompletas."

display(df_com_indice)

# Calcular estatísticas
tempos = df_com_indice["Tempo (ms)"]

print(f"Tempo médio: {tempos.mean():.3f} ms")
print(f"Mediana: {tempos.median():.3f} ms")
print(f"Menor tempo: {tempos.min():.3f} ms")
print(f"Maior tempo: {tempos.max():.3f} ms")

Índice encontrado!
CREATE INDEX idx_mov_componente_momento_id ON industria.movimentacoes USING btree (componente_id, momento DESC, id DESC)



,Execução,Volume,Tempo (ms)
0,1,2000000,435.234
1,2,2000000,404.106
2,3,2000000,391.140
3,4,2000000,380.321
4,5,2000000,394.335


Tempo médio: 401.027 ms
Mediana: 394.335 ms
Menor tempo: 380.321 ms
Maior tempo: 435.234 ms


Análise do plano de execução após a criação do índice

In [42]:
from pathlib import Path

# Recuperar o último plano medido com índice
texto_plano = "\n".join(plano_com_indice)

# Salvar o plano completo
pasta = raiz / "evidencias" / "INC-001"
pasta.mkdir(parents=True, exist_ok=True)

arquivo = pasta / "07_plano_com_indice.txt"
arquivo.write_text(texto_plano, encoding="utf-8")

# Exibir o plano
print(texto_plano)

print("\nPlano salvo em:", arquivo)

Limit  (cost=23103.72..23103.97 rows=100 width=128) (actual time=394.294..394.303 rows=100.00 loops=1)
  Buffers: shared hit=16 read=38505
  ->  Sort  (cost=23103.72..23128.71 rows=9998 width=128) (actual time=394.293..394.297 rows=100.00 loops=1)
        Sort Key: m.momento DESC, m.id DESC
        Sort Method: top-N heapsort  Memory: 51kB
        Buffers: shared hit=16 read=38505
        ->  Nested Loop  (cost=328.04..22721.60 rows=9998 width=128) (actual time=28.030..319.110 rows=706404.00 loops=1)
              Buffers: shared hit=16 read=38505
              ->  Nested Loop  (cost=327.90..22488.47 rows=9998 width=78) (actual time=28.019..169.511 rows=706404.00 loops=1)
                    Buffers: shared hit=4 read=38505
                    ->  Seq Scan on componentes c  (cost=0.00..6.50 rows=1 width=8) (actual time=0.004..0.042 rows=1.00 loops=1)
                          Filter: ((codigo)::text = 'COR-002'::text)
                          Rows Removed by Filter: 199
              

### Interpretação do desempenho após a criação do índice

Após a criação do índice composto `idx_mov_componente_momento_id`, foi realizada uma nova análise de desempenho com 2.000.000 de movimentações.

O PostgreSQL passou a utilizar o índice por meio de `Bitmap Index Scan`, seguido de `Bitmap Heap Scan`. Entretanto, a consulta precisou processar aproximadamente 706.404 movimentações do componente COR-002 antes de retornar os 100 registros solicitados.

A mediana do tempo de execução aumentou de 545,096 ms para 1.174,924 ms, representando uma piora de aproximadamente 115,5%.

O plano de execução também apresentou uma diferença significativa entre a quantidade estimada de registros (9.998) e a quantidade efetivamente encontrada (706.404), o que pode ter contribuído para a escolha de uma estratégia pouco eficiente.

**Conclusão parcial:** embora o índice tenha sido utilizado, ele não melhorou o desempenho da consulta nas condições testadas. Isso demonstra que a existência de um índice não garante consultas mais rápidas, especialmente quando os filtros selecionam uma grande quantidade de registros.

Como próxima etapa, será avaliada uma reestruturação da consulta SQL, buscando reduzir a quantidade de registros processados antes da aplicação do limite de 100 movimentações.

Teste de uma consulta otimizada com seleção antecipada dos registros

In [43]:
# Consulta alternativa para o histórico
consulta_otimizada = """
WITH ultimas AS MATERIALIZED (
    SELECT
        m.id,
        m.momento,
        m.deposito_id,
        m.tipo,
        m.quantidade,
        m.observacao
    FROM industria.movimentacoes m
    WHERE m.componente_id = (
        SELECT c.id
        FROM industria.componentes c
        WHERE c.codigo = %s
    )
      AND m.momento >= %s
      AND m.momento < %s
    ORDER BY m.momento DESC, m.id DESC
    LIMIT %s
)
SELECT
    u.id,
    u.momento,
    d.codigo,
    u.tipo,
    u.quantidade,
    u.observacao
FROM ultimas u
JOIN industria.depositos d
    ON d.id = u.deposito_id
ORDER BY u.momento DESC, u.id DESC;
"""

# Consulta original, sem EXPLAIN
consulta_original = """
SELECT
    m.id,
    m.momento,
    d.codigo,
    m.tipo,
    m.quantidade,
    m.observacao
FROM industria.movimentacoes m
JOIN industria.componentes c
    ON c.id = m.componente_id
JOIN industria.depositos d
    ON d.id = m.deposito_id
WHERE c.codigo = %s
  AND m.momento >= %s
  AND m.momento < %s
ORDER BY m.momento DESC, m.id DESC
LIMIT %s;
"""

parametros = (
    componente,
    data_inicio,
    fim_exclusivo,
    limite
)

with conectar_testes() as conn:
    with conn.cursor() as cur:

        cur.execute("SELECT current_database()")
        assert cur.fetchone()[0] == "pbl_industria_testes"

        # Executar a consulta original
        cur.execute(consulta_original, parametros)
        resultado_original = cur.fetchall()

        # Executar a alternativa otimizada
        cur.execute(consulta_otimizada, parametros)
        resultado_otimizado = cur.fetchall()

print("Registros originais:", len(resultado_original))
print("Registros otimizados:", len(resultado_otimizado))

iguais = resultado_original == resultado_otimizado

print("Resultados idênticos:", iguais)

assert iguais, "As consultas retornaram resultados diferentes!"

Registros originais: 100
Registros otimizados: 100
Resultados idênticos: True


Medição de desempenho da consulta SQL otimizada

In [44]:
import re
import pandas as pd

# Consulta otimizada com análise de desempenho
sql_explain_otimizada = (
    "EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)\n"
    + consulta_otimizada.strip().rstrip(";")
)

resultados_otimizados = []

with conectar_testes() as conn:
    with conn.cursor() as cur:

        # Confirmar o banco utilizado
        cur.execute("""
            SELECT
                current_database(),
                COUNT(*)
            FROM industria.movimentacoes;
        """)

        banco, total = cur.fetchone()

        assert banco == "pbl_industria_testes"
        assert total == 2000000

        # Confirmar equivalência com a consulta original
        cur.execute(consulta_original, parametros)
        resultado_original = cur.fetchall()

        cur.execute(consulta_otimizada, parametros)
        resultado_otimizado = cur.fetchall()

        assert resultado_original == resultado_otimizado, (
            "As consultas retornaram resultados diferentes!"
        )

        print("Equivalência dos resultados: OK")

        # Executar cinco medições
        for tentativa in range(1, 6):

            cur.execute(
                sql_explain_otimizada,
                parametros
            )

            plano = [
                linha[0]
                for linha in cur.fetchall()
            ]

            # Extrair o tempo real
            for linha in plano:
                resultado = re.search(
                    r"Execution Time:\s*([\d.]+)\s*ms",
                    linha
                )

                if resultado:
                    tempo = float(resultado.group(1))

                    resultados_otimizados.append({
                        "Execução": tentativa,
                        "Volume": 2000000,
                        "Tempo (ms)": tempo
                    })

                    break

            # Guardar o último plano de execução
            if tentativa == 5:
                plano_sql_otimizado = plano

# Organizar resultados
df_otimizado = pd.DataFrame(resultados_otimizados)

assert len(df_otimizado) == 5, "Medições incompletas."

display(df_otimizado)

# Calcular estatísticas
tempos = df_otimizado["Tempo (ms)"]

print(f"Tempo médio: {tempos.mean():.3f} ms")
print(f"Mediana: {tempos.median():.3f} ms")
print(f"Menor tempo: {tempos.min():.3f} ms")
print(f"Maior tempo: {tempos.max():.3f} ms")

Equivalência dos resultados: OK


,Execução,Volume,Tempo (ms)
0,1,2000000,0.121
1,2,2000000,0.121
2,3,2000000,0.121
3,4,2000000,0.112
4,5,2000000,0.111


Tempo médio: 0.117 ms
Mediana: 0.121 ms
Menor tempo: 0.111 ms
Maior tempo: 0.121 ms


Análise do plano de execução da consulta otimizada

In [45]:
from pathlib import Path

# Recuperar o plano da consulta otimizada
texto_plano = "\n".join(plano_sql_otimizado)

# Criar a pasta de evidências
pasta = raiz / "evidencias" / "INC-001"
pasta.mkdir(parents=True, exist_ok=True)

# Salvar o plano completo em TXT
arquivo = pasta / "09_plano_consulta_otimizada.txt"

arquivo.write_text(
    texto_plano,
    encoding="utf-8"
)

# Exibir o plano de execução
print(texto_plano)

print("\nPlano salvo em:", arquivo)

Nested Loop  (cost=358.18..364.12 rows=100 width=608) (actual time=0.026..0.098 rows=100.00 loops=1)
  Buffers: shared hit=119
  CTE ultimas
    ->  Limit  (cost=6.93..358.04 rows=100 width=78) (actual time=0.023..0.051 rows=100.00 loops=1)
          Buffers: shared hit=107
          InitPlan 1
            ->  Seq Scan on componentes c  (cost=0.00..6.50 rows=1 width=8) (actual time=0.003..0.011 rows=1.00 loops=1)
                  Filter: ((codigo)::text = 'COR-002'::text)
                  Rows Removed by Filter: 199
                  Buffers: shared hit=4
          ->  Index Scan using idx_mov_componente_momento_id on movimentacoes m  (cost=0.43..35104.23 rows=9998 width=78) (actual time=0.022..0.045 rows=100.00 loops=1)
                Index Cond: ((componente_id = (InitPlan 1).col1) AND (momento >= '2026-08-01'::date) AND (momento < '2026-11-11'::date))
                Index Searches: 1
                Buffers: shared hit=107
  ->  CTE Scan on ultimas u  (cost=0.00..2.00 rows=100 w

### Interpretação do plano da consulta otimizada

A consulta reestruturada apresentou uma melhoria expressiva
de desempenho em relação à implementação original.

O plano de execução confirmou a utilização do índice
idx_mov_componente_momento_id por meio de Index Scan.

Com a seleção antecipada dos registros, o PostgreSQL
localizou apenas as 100 movimentações necessárias,
evitando o processamento de aproximadamente 706 mil
movimentações observado anteriormente.

A mediana de execução foi reduzida de 545,096 ms,
na consulta original sem o índice composto, para
0,449 ms na consulta reestruturada com índice.

Isso representa uma redução aproximada de 99,92%
no tempo mediano de execução nas condições testadas.

A equivalência dos 100 registros retornados também
foi confirmada por comparação direta.

Os resultados indicam que a combinação do índice
composto com a reestruturação da consulta é uma
solução promissora para o INC-001.

A validação definitiva deverá incluir outros componentes,
períodos, volumes e testes de carga, considerando também
o custo de manutenção do índice.

Comparação final dos cenários de desempenho

In [46]:
import pandas as pd

# Tempos obtidos nos experimentos anteriores (ms)
medicoes = {
    "Original - 500 mil": [
        182.197, 164.983, 155.520, 158.499, 164.669
    ],
    "Original - 1 milhão": [
        289.437, 269.666, 300.610, 289.580, 279.304
    ],
    "Original - 2 milhões": [
        619.477, 547.130, 534.991, 545.096, 538.244
    ],
    "Original com índice - 2 milhões": [
        1266.679, 1217.149, 1134.517, 1174.924, 1151.958
    ],
    "Otimizada com índice - 2 milhões": [
        0.513, 0.442, 0.449, 0.501, 0.374
    ]
}

volumes = [
    500000,
    1000000,
    2000000,
    2000000,
    2000000
]

# Consolidar as medianas
comparativo = pd.DataFrame({
    "Cenário": list(medicoes.keys()),
    "Volume": volumes,
    "Mediana (ms)": [
        pd.Series(tempos).median()
        for tempos in medicoes.values()
    ]
})

display(comparativo.round(3))

# Comparar a consulta original e a otimizada
tempo_original = comparativo.loc[2, "Mediana (ms)"]
tempo_otimizado = comparativo.loc[4, "Mediana (ms)"]

reducao = (
    (tempo_original - tempo_otimizado)
    / tempo_original
) * 100

aceleracao = tempo_original / tempo_otimizado

print(f"Redução de tempo: {reducao:.2f}%")
print(f"Fator de aceleração: {aceleracao:.2f}x")

,Cenário,Volume,Mediana (ms)
0,Original - 500 mil,500000,164.669
1,Original - 1 milhão,1000000,289.437
2,Original - 2 milhões,2000000,545.096
3,Original com índice - 2 milhões,2000000,1174.924
4,Otimizada com índice - 2 milhões,2000000,0.449


Redução de tempo: 99.92%
Fator de aceleração: 1214.02x


# Ficha técnica | INC-001

## 1. Hipótese e resultado esperado

O cliente relatou lentidão na consulta ao histórico de movimentações conforme o crescimento da base de dados, principalmente em períodos longos.

A hipótese investigada foi a de que a ausência de índices adequados, associada à estrutura da consulta SQL, provocava o processamento excessivo de registros.

O comportamento esperado é que o sistema consiga consultar o histórico de um componente com desempenho adequado, mesmo com o crescimento da quantidade de movimentações.

## 2. Procedimento de investigação

Os experimentos foram realizados com PostgreSQL 18, Python e Jupyter Notebook, utilizando exclusivamente o banco `pbl_industria_testes`.

Foi utilizado o componente `COR-002`, com período de 01/08/2026 a 10/11/2026 e limite de 100 registros.

Foram realizados os seguintes procedimentos:

1. Análise dos índices existentes na tabela `industria.movimentacoes`.
2. Execução de `EXPLAIN (ANALYZE, BUFFERS)` para identificar as operações realizadas pelo PostgreSQL.
3. Simulação controlada do crescimento da base de 500.000 para 1.000.000 e posteriormente 2.000.000 de movimentações, mediante duplicação de registros na cópia de testes.
4. Atualização das estatísticas com `ANALYZE`.
5. Realização de cinco medições em cada cenário.
6. Criação do índice composto `idx_mov_componente_momento_id`.
7. Nova análise do plano de execução da consulta original.
8. Reestruturação experimental da consulta SQL, aplicando o limite de registros antes da junção com a tabela de depósitos.
9. Verificação da equivalência dos 100 registros retornados.
10. Comparação final das medianas dos tempos de execução.

## 3. Evidências e resultados

| Cenário | Mediana |
|---|---:|
| Consulta original com 500 mil registros | 164,669 ms |
| Consulta original com 1 milhão de registros | 289,437 ms |
| Consulta original com 2 milhões de registros | 545,096 ms |
| Consulta original com índice composto, 2 milhões | 1.174,924 ms |
| Consulta reestruturada com índice, 2 milhões | 0,449 ms |

O plano inicial apresentou `Parallel Seq Scan`, realizando leituras extensas da tabela de movimentações.

Após a criação do índice, a consulta original passou a utilizar `Bitmap Index Scan` e `Bitmap Heap Scan`, mas apresentou desempenho inferior, processando aproximadamente 706 mil movimentações antes de retornar os 100 registros.

Na consulta reestruturada, o PostgreSQL utilizou `Index Scan`, selecionando 100 movimentações antecipadamente. O plano analisado apresentou tempo de execução de 0,374 ms.

A mediana da consulta reestruturada com índice representou uma redução aproximada de 99,92% em relação à consulta original sem o índice composto, no cenário de 2 milhões de registros.

## 4. Conclusão

O incidente foi confirmado nas condições experimentais: o crescimento simulado da tabela resultou em aumento significativo no tempo de execução.

A investigação demonstrou que a simples criação de um índice não foi suficiente para melhorar o desempenho da consulta original.

A combinação do índice composto com a reestruturação SQL apresentou melhoria expressiva, reduzindo a quantidade de movimentações processadas antes da aplicação do limite.

Entretanto, os resultados foram obtidos em ambiente local, com dados sintéticos gerados pela duplicação de registros. Os tempos não devem ser interpretados como garantia de desempenho em produção.

## 5. Solução proposta e validação

Recomenda-se avaliar a adoção conjunta de:

- Índice composto sobre `componente_id`, `momento DESC` e `id DESC`.
- Reestruturação da consulta do histórico para selecionar antecipadamente as movimentações necessárias.
- Monitoramento dos planos de execução e atualização das estatísticas do PostgreSQL.

A solução deve ser validada com diferentes componentes, períodos e volumes de dados, incluindo testes de equivalência dos resultados e avaliação do custo de manutenção do índice em operações de escrita.

A implantação deverá ser considerada bem-sucedida quando preservar os resultados funcionais da consulta e apresentar ganhos consistentes de desempenho em condições representativas do ambiente de produção.

## Evidências complementares — INC-001

Esta seção reúne os registros visuais e os planos de execução armazenados em `evidencias/INC-001`. As imagens permanecem na pasta original e são exibidas aqui por caminhos relativos ao notebook.

### Medições com 1 milhão de movimentações

![Medições com 1 milhão](../evidencias/INC-001/02_medicoes_1_milhao.png)

### Medições com 2 milhões de movimentações

![Medições com 2 milhões](../evidencias/INC-001/04_medicoes_2_milhoes.png)

### Plano de execução original com 2 milhões de movimentações

![Plano original com 2 milhões](../evidencias/INC-001/05_plano_2_milhoes.png)

### Medições após a criação do índice

![Medições com índice](../evidencias/INC-001/06_medicoes_com_indice.png)

#### Plano de execução com índice

**Plano de execução com índice**

```text
Limit  (cost=23103.53..23103.78 rows=100 width=128) (actual time=1151.857..1151.882 rows=100.00 loops=1)
  Buffers: shared hit=16 read=38504
  ->  Sort  (cost=23103.53..23128.53 rows=9998 width=128) (actual time=1151.854..1151.866 rows=100.00 loops=1)
        Sort Key: m.momento DESC, m.id DESC
        Sort Method: top-N heapsort  Memory: 51kB
        Buffers: shared hit=16 read=38504
        ->  Nested Loop  (cost=328.04..22721.42 rows=9998 width=128) (actual time=83.750..854.193 rows=706404.00 loops=1)
              Buffers: shared hit=16 read=38504
              ->  Nested Loop  (cost=327.90..22488.28 rows=9998 width=78) (actual time=83.714..447.878 rows=706404.00 loops=1)
                    Buffers: shared hit=4 read=38504
                    ->  Seq Scan on componentes c  (cost=0.00..6.50 rows=1 width=8) (actual time=0.016..0.147 rows=1.00 loops=1)
                          Filter: ((codigo)::text = 'COR-002'::text)
                          Rows Removed by Filter: 199
                          Buffers: shared hit=2 read=2
                    ->  Bitmap Heap Scan on movimentacoes m  (cost=327.90..22381.80 rows=9998 width=86) (actual time=83.690..283.564 rows=706404.00 loops=1)
                          Recheck Cond: ((componente_id = c.id) AND (momento >= '2026-08-01'::date) AND (momento < '2026-11-11'::date))
                          Heap Blocks: exact=35021
                          Buffers: shared hit=2 read=38502
                          ->  Bitmap Index Scan on idx_mov_componente_momento_id  (cost=0.00..325.40 rows=9998 width=0) (actual time=70.824..70.825 rows=706404.00 loops=1)
                                Index Cond: ((componente_id = c.id) AND (momento >= '2026-08-01'::date) AND (momento < '2026-11-11'::date))
                                Index Searches: 1
                                Buffers: shared hit=2 read=3481
              ->  Memoize  (cost=0.14..0.16 rows=1 width=66) (actual time=0.000..0.000 rows=1.00 loops=706404)
                    Cache Key: m.deposito_id
                    Cache Mode: logical
                    Hits: 706398  Misses: 6  Evictions: 0  Overflows: 0  Memory Usage: 1kB
                    Buffers: shared hit=12
                    ->  Index Scan using depositos_pkey on depositos d  (cost=0.13..0.15 rows=1 width=66) (actual time=0.005..0.005 rows=1.00 loops=6)
                          Index Cond: (id = m.deposito_id)
                          Index Searches: 6
                          Buffers: shared hit=12
Planning:
  Buffers: shared hit=4 read=8
Planning Time: 0.747 ms
Execution Time: 1151.958 ms
```

### Medições da consulta otimizada

![Medições da consulta otimizada](../evidencias/INC-001/08_medicoes_consulta_otimizada.png)

#### Plano de execução da consulta otimizada

**Plano de execução da consulta otimizada**

```text
Nested Loop  (cost=358.26..364.21 rows=100 width=608) (actual time=0.087..0.326 rows=100.00 loops=1)
  Buffers: shared hit=119
  CTE ultimas
    ->  Limit  (cost=6.93..358.12 rows=100 width=78) (actual time=0.072..0.150 rows=100.00 loops=1)
          Buffers: shared hit=107
          InitPlan 1
            ->  Seq Scan on componentes c  (cost=0.00..6.50 rows=1 width=8) (actual time=0.013..0.035 rows=1.00 loops=1)
                  Filter: ((codigo)::text = 'COR-002'::text)
                  Rows Removed by Filter: 199
                  Buffers: shared hit=4
          ->  Index Scan using idx_mov_componente_momento_id on movimentacoes m  (cost=0.43..35112.66 rows=9998 width=78) (actual time=0.071..0.138 rows=100.00 loops=1)
                Index Cond: ((componente_id = (InitPlan 1).col1) AND (momento >= '2026-08-01'::date) AND (momento < '2026-11-11'::date))
                Index Searches: 1
                Buffers: shared hit=107
  ->  CTE Scan on ultimas u  (cost=0.00..2.00 rows=100 width=558) (actual time=0.076..0.214 rows=100.00 loops=1)
        Storage: Memory  Maximum Storage: 27kB
        Buffers: shared hit=107
  ->  Memoize  (cost=0.14..0.28 rows=1 width=66) (actual time=0.000..0.000 rows=1.00 loops=100)
        Cache Key: u.deposito_id
        Cache Mode: logical
        Hits: 94  Misses: 6  Evictions: 0  Overflows: 0  Memory Usage: 1kB
        Buffers: shared hit=12
        ->  Index Scan using depositos_pkey on depositos d  (cost=0.13..0.27 rows=1 width=66) (actual time=0.002..0.002 rows=1.00 loops=6)
              Index Cond: (id = u.deposito_id)
              Index Searches: 6
              Buffers: shared hit=12
Planning Time: 0.280 ms
Execution Time: 0.374 ms
```

### Comparação final dos cenários

![Comparação final](../evidencias/INC-001/10_comparacao_final.png)

As imagens e os planos acima são evidências dos resultados registrados durante os experimentos. Os tempos podem variar entre execuções, dependendo do estado do banco, cache e ambiente de execução.